# Sandbox — Silver Payments

Exploração por trás de `silver_rules.tratar_payments`. Foco: normalizar `payment_type` e **castar os campos numéricos** (`payment_sequential` e `payment_installments` para int, `payment_value` para double).

In [ ]:
from utils.spark_utils import create_spark_session
from pyspark.sql.functions import col, trim, lower, current_timestamp

spark = create_spark_session('Sandbox-Silver-Payments')

df_bronze = spark.read.parquet('s3a://bronze/olist/payments')
df_bronze.show(5, truncate=False)
df_bronze.printSchema()

Vamos ver os valores distintos de `payment_type` para decidir a normalização (esperamos `credit_card`, `boleto`, `voucher`, `debit_card`).

In [ ]:
df_bronze.select('payment_type').distinct().show()

In [ ]:
df_silver = (df_bronze
    .withColumn('order_id', trim(col('order_id')))
    .withColumn('payment_sequential', col('payment_sequential').cast('int'))
    .withColumn('payment_type', lower(trim(col('payment_type'))))
    .withColumn('payment_installments', col('payment_installments').cast('int'))
    .withColumn('payment_value', col('payment_value').cast('double'))
    .withColumn('dt_criacao_silver', current_timestamp()))

df_silver.printSchema()
df_silver.show(5, truncate=False)

**Lógica promovida para `silver_rules.tratar_payments`.**

In [ ]:
spark.stop()